# 08 · Spark and Beam

*The same station job in two more frameworks, and what each one is built for.*

Notebook 07 ran the pipeline on Ray. This notebook takes the two steps that matter and writes them in **Apache Spark** and **Apache Beam**. The point isn't a benchmark on our small data. Each framework was designed around a different idea of what a data job is, and those ideas decide where each one is easy and where it's awkward. Writing the same job in each is the quickest way to feel that.

- **Spark** thinks in tables. You describe a query over DataFrames (select, join, group) and its optimizer turns it into a plan of stages split at **shuffles**, the points where rows must move between machines because they're grouped by a new key. It grew out of large batch analytics and SQL.
- **Beam** thinks in collections flowing through a graph. A **PCollection** is a dataset that may be finite (batch) or never-ending (stream), and every element carries an **event time**. The same pipeline code runs on different engines called **runners**: Google Cloud Dataflow, Flink, Spark or a local DirectRunner. It grew out of Google's streaming work and its big idea is that batch is just a stream that ends.

Two steps from our pipeline show the difference best. One is the **neighbour difference**, which compares each station with others at the same hour and so needs a join across stations (a shuffle). The other is **late data**: a station-hour that arrives after its day has already been checked.

**Read first:** Spark's [SQL and DataFrames guide](https://spark.apache.org/docs/latest/sql-programming-guide.html) (overview and "Getting started"), Beam's [Programming guide](https://beam.apache.org/documentation/programming-guide/) sections 1–3 and 7 (windowing), and the classic [Streaming 101](https://www.oreilly.com/radar/the-world-beyond-batch-streaming-101/) essay. About 60 minutes.

Spark needs a Java 17 or 21 runtime (`brew install openjdk@21` on macOS).

In [ ]:
# Setup: identical in every notebook. Data lives in ../data, which git ignores.
import os
DATA_DIR = os.path.abspath(os.path.join(os.getcwd(), "..", "data"))
os.makedirs(DATA_DIR, exist_ok=True)
print("Data directory:", DATA_DIR)

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import time
import pandas as pd
from pathlib import Path
from wxlab import qa

STD = Path(DATA_DIR) / "standardized" / "isd"
points = pd.read_parquet(Path(DATA_DIR) / "reference" / "station_points.parquet")

## Part 1 · Spark

### A session and a table

A `SparkSession` is the entry point. Locally it runs the driver and the workers as threads in one Java process. Reading the standardized Parquet tree gives a DataFrame. Spark reads only the files and columns the query needs, and the `year=` folders become a column it can prune on (**partition pruning**).

In [ ]:
from pyspark.sql import SparkSession, functions as F, Window

spark = (SparkSession.builder.master("local[8]").appName("wxlab")
         .config("spark.sql.session.timeZone", "UTC")
         .config("spark.sql.shuffle.partitions", "16")      # the default of 200 is sized for clusters
         .getOrCreate())
obs = spark.read.parquet(str(STD)).where(F.col("year").between(2021, 2024))
obs = obs.select("station_id", "time", "air_temperature", "year").where(F.col("air_temperature").isNotNull())
print(obs.count(), "station-hours")

### The neighbour difference as a join

In pandas notebook 05 computed, for each station-hour, the station's temperature minus the median of its neighbours at the same hour. In Spark that's a join: station pairs from a small table, joined to the observations twice, on station and on time. Spark broadcasts the small pairs table to every worker, but the two sides of the time join must be **shuffled** so equal hours land together.

In [ ]:
nbrs = qa.neighbours(points)                         # station → its neighbours, as in notebook 05
pairs = spark.createDataFrame([(s, t) for s, ns in nbrs.items() for t in ns], ["station_id", "nbr_id"])
o = obs.alias("o"); nb = obs.alias("nb")
joined = (o.join(F.broadcast(pairs), "station_id")
           .join(nb, (F.col("nbr_id") == F.col("nb.station_id")) & (F.col("o.time") == F.col("nb.time")))
           .select(F.col("o.station_id").alias("station_id"), F.col("o.time").alias("time"),
                   F.col("o.air_temperature").alias("t"), F.col("nb.air_temperature").alias("t_nbr")))
nbr_diff = joined.groupBy("station_id", "time").agg((F.first("t") - F.percentile_approx("t_nbr", 0.5)).alias("nbr_diff"))
nbr_diff.explain(mode="formatted")

*Draft note: once run, point at the `Exchange` operators in the plan. Each one is a shuffle: rows written to disk by key and read back by other tasks. Count them. The broadcast join has none. This is where Spark's maturity shows: it plans shuffles, spills to disk when memory runs out, and on a cluster it has a shuffle service that survives a worker dying.*

In [ ]:
t0 = time.time()
out = nbr_diff.toPandas()
print(f"{len(out):,} rows in {time.time() - t0:.0f} s"); out.nbr_diff.describe()

### Window functions and reusing pandas code

Many QA checks look along one station's own record, like "this hour versus the 24 before it". Spark's **window functions** do that without a shuffle beyond grouping by station. When the logic is easier in pandas, `applyInPandas` hands each group to a Python function as a pandas DataFrame, so the phase 1 code can be reused at the cost of moving data between Java and Python (through Apache Arrow).

In [ ]:
w = Window.partitionBy("station_id").orderBy(F.col("time").cast("long")).rangeBetween(-24 * 3600, -1)
stuck_like = obs.withColumn("prev_max", F.max("air_temperature").over(w)) \
                .withColumn("prev_min", F.min("air_temperature").over(w)) \
                .withColumn("flat_24h", F.col("prev_max") == F.col("prev_min"))

def robust_z(pdf: pd.DataFrame) -> pd.DataFrame:
    med = pdf.air_temperature.median(); mad = (pdf.air_temperature - med).abs().median() * 1.4826
    return pdf.assign(z=(pdf.air_temperature - med) / (mad or 1.0))[["station_id", "time", "z"]]

z = obs.groupBy("station_id").applyInPandas(robust_z, schema="station_id string, time timestamp, z double")
print(stuck_like.where("flat_24h").count(), "hours flat for the previous 24 h;", z.where(F.abs("z") > 6).count(), "with |z| > 6")

In [ ]:
spark.stop()

## Part 2 · Beam

### The standardize step as a Beam pipeline

A Beam pipeline is a graph of **transforms** applied to PCollections. `Create` makes a PCollection from a list, `Map` and `ParDo` apply a function to each element, and the pipeline runs when the `with` block exits. On the local DirectRunner it's slow and meant for testing. The same code would run on Dataflow or Flink by changing one option.

In [ ]:
import apache_beam as beam
from apache_beam.options.pipeline_options import PipelineOptions
from wxlab import pipeline as pl

RAW, OUT = Path(DATA_DIR) / "raw" / "isd", Path(DATA_DIR) / "work" / "beam" / "standardized"
units = pl.units_for(sorted(p.stem for p in (STD / "year=2023").glob("*.parquet"))[:20], [2023])

opts = PipelineOptions(runner="DirectRunner", direct_num_workers=4, direct_running_mode="multi_processing")
with beam.Pipeline(options=opts) as p:
    (p | "units" >> beam.Create(units)
       | "standardize" >> beam.Map(pl.run_unit, RAW, OUT)
       | "ok?" >> beam.Map(lambda row: (row["status"], 1))
       | "count" >> beam.CombinePerKey(sum)
       | "print" >> beam.Map(print))

The unit function is the same one Ray ran. What Beam adds isn't visible in a batch job this small. It's the model for time.

### Event time, windows and late data

In a live feed each station-hour is an element with an **event time** (when it was measured), and it arrives at some **processing time** (when the pipeline sees it). Beam groups elements into **windows** by event time, for example one window per UTC day for daily checks. It tracks a **watermark**, its estimate of how far event time has progressed, to decide when a window is complete enough to emit. Data that arrives after the watermark has passed its window is **late**. With `allowed_lateness` Beam re-fires the window with the late data included, which is exactly "verdicts change when late data lands" from notebook 06.

`TestStream` lets us script a stream with controlled arrival times on the DirectRunner: two days of readings from one station, then the watermark moves past day one, then a late reading for day one arrives.

In [ ]:
from apache_beam.testing.test_stream import TestStream
from apache_beam.transforms import window, trigger
from apache_beam.options.pipeline_options import StandardOptions

d1, d2 = pd.Timestamp("2024-02-05", tz="UTC").timestamp(), pd.Timestamp("2024-02-06", tz="UTC").timestamp()
stream = (TestStream()
          .add_elements([window.TimestampedValue(("KSFO", 12.0), d1 + 3600 * h) for h in range(0, 24, 6)])
          .add_elements([window.TimestampedValue(("KSFO", 14.0), d2 + 3600 * h) for h in range(0, 24, 6)])
          .advance_watermark_to(d2 + 3600)                  # day one is now "complete"
          .add_elements([window.TimestampedValue(("KSFO", 30.0), d1 + 3600 * 23)])   # a late, odd reading for day one
          .advance_watermark_to_infinity())

opts = PipelineOptions(); opts.view_as(StandardOptions).streaming = True
with beam.Pipeline(options=opts) as p:
    (p | stream
       | beam.WindowInto(window.FixedWindows(24 * 3600), allowed_lateness=2 * 24 * 3600,
                         trigger=trigger.AfterWatermark(late=trigger.AfterCount(1)),
                         accumulation_mode=trigger.AccumulationMode.ACCUMULATING)
       | beam.CombinePerKey(lambda xs: (len(list(xs)), max(xs)) if not isinstance(xs, list) else (len(xs), max(xs)))
       | beam.Map(lambda kv, w=beam.DoFn.WindowParam, pane=beam.DoFn.PaneInfoParam:
                  print(f"{w.start.to_utc_datetime():%Y-%m-%d} {pane.timing}: {kv}")))

*Draft note: once run, expect day one to fire ON_TIME with 4 readings, then fire again LATE with 5 readings and the new maximum. Explain that the downstream daily verdict for day one is now version 2. Fix the combine lambda if Beam passes an iterable: `beam.combiners.CountCombineFn` plus `MaxCombineFn` via `CombineGlobally` per key is the cleaner form.*

That firing pattern is the piece Spark's batch model and Ray don't have built in. Spark has Structured Streaming with event-time watermarks too, but its model is "a table that keeps growing" and late updates are handled per query. Beam made windows, triggers and accumulation first-class, and Flink implements the same model natively.

### Beam for grids

Beam also underpins **xarray-beam**, which runs xarray computations as Beam pipelines over chunks of big arrays. It's how WeatherBench 2 computes its scores over terabytes of forecasts. Notebook 11 uses it to rechunk the HRRR cube.

## How the three compare

| | Ray | Spark | Beam |
|---|---|---|---|
| Core idea | Python tasks and actors | Tables and SQL-style queries | Collections over event time, any runner |
| Natural fit | ML pipelines, mixed and stateful work, GPUs | Large joins, aggregations and shuffles, SQL | Streaming with windows and late data; batch on the same code |
| Awkward at | Big shuffles and joins, event-time windows | Arbitrary Python per row (crosses into Python), stateful actors | Interactive work; local runner is slow; debugging through a runner |
| Fault tolerance | Task retries from lineage; actors need care | Stage retries; shuffle files survive worker loss | Delegated to the runner (Dataflow, Flink checkpoints) |
| In this lab | Notebook 07: units, manifest, actors | Neighbour join, windows, pandas UDF | Same unit function; late data re-fires a day |

*Draft note: add a small timing row for the standardize step on the same 20 station-years in all three, labelled as illustrative only.*

## What surprised me

*Draft: fill in after running.*

## Check yourself

<details><summary>Why does the neighbour difference need a shuffle but the 24-hour window doesn't?</summary>
The window only looks within one station, and the data can be partitioned by station once. The neighbour difference needs different stations' readings for the same hour together, which needs a regroup by time.
</details>

<details><summary>What does allowed_lateness change about a daily window?</summary>
It keeps the window's state after the watermark passes, so late readings within the allowance trigger a new result for that day instead of being dropped.
</details>

<details><summary>When would you choose Beam over Spark for this pipeline?</summary>
When the main path is a live feed where data arrives late or out of order and results must be updated by event time, and when you want the same code for the historical backfill.
</details>

New terms from this notebook are in [`docs/glossary.md`](../docs/glossary.md).